In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score


df = pd.read_csv('modified_digital_media_with_business_segment.csv')


target_col = 'performance_tier_index'

X = df.drop(columns=[
    target_col,
    'campaign_id'
], errors='ignore')

y = df[target_col]


if y.dtype == 'object':
    y = y.astype('category').cat.codes

from sklearn.preprocessing import OrdinalEncoder

cat_cols = X.select_dtypes(include='object').columns

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X[cat_cols] = encoder.fit_transform(X[cat_cols])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_model = XGBClassifier(
    subsample=1.0,
    reg_lambda=2,
    reg_alpha=1,
    n_estimators=1000,
    min_child_weight=3,
    max_depth=2,
    learning_rate=0.02,
    gamma=0.5,
    colsample_bytree=1.0,
    
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss',
    enable_categorical=True
)

from sklearn.model_selection import StratifiedKFold, cross_val_score


kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1
)

val_acc = cv_scores.mean()

print("\nCV Scores (5-Fold):", cv_scores)
best_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)


y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)


train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

train_precision = precision_score(y_train, y_train_pred, average='macro')
test_precision = precision_score(y_test, y_test_pred, average='macro')

train_recall = recall_score(y_train, y_train_pred, average='macro')
test_recall = recall_score(y_test, y_test_pred, average='macro')

print("\n" + "="*65)
print(f"{'Metric':<20} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10}")
print("-"*65)

print(f"{'Training':<20} | {train_acc:.4f}   | {train_precision:.4f}   | {train_recall:.4f}")
print(f"{'Validation (CV)':<20} | {val_acc:.4f}   | {'N/A':<10} | {'N/A':<10}")
print(f"{'Test':<20} | {test_acc:.4f}   | {test_precision:.4f}   | {test_recall:.4f}")

print("="*65)


CV Scores (5-Fold): [0.92997199 0.85714286 0.90756303 0.89915966 0.87921348]


c:\Users\Saquire\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:28:47] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Metric               | Accuracy   | Precision  | Recall    
-----------------------------------------------------------------
Training             | 0.9608   | 0.9670   | 0.9583
Validation (CV)      | 0.8946   | N/A        | N/A       
Test                 | 0.9195   | 0.9222   | 0.9238


In [2]:
import joblib

joblib.dump(best_model, "xgb_clf.pkl")
joblib.dump(encoder, "encoder.pkl")
joblib.dump(X.columns.tolist(), "features.pkl")
joblib.dump(cat_cols.tolist(), "cat_cols.pkl")

['cat_cols.pkl']